# 03 — Apply DQX Checks & Quarantine Bad Rows

The main event. We:

1. Load the hand-curated `checks.yml`.
2. Define a couple of extra rules in **Python** (DQRowRule / DQDatasetRule) — same engine, two definition styles.
3. Validate both check sets.
4. Run `apply_checks_and_split` → `(valid_df, invalid_df)`.
5. Write valid rows to **silver**, invalid rows to **quarantine** (with `_errors` / `_warnings`).

Pattern is exactly what DQX recommends in the *Apply Checks* guide.

In [1]:
from databricks.connect import DatabricksSession
from databricks.sdk import WorkspaceClient
from databricks.labs.dqx.engine import DQEngine
from databricks.labs.dqx.rule import DQRowRule, DQDatasetRule, Criticality
from databricks.labs.dqx import check_funcs
from dotenv import load_dotenv
import os, yaml
from pathlib import Path

load_dotenv()
spark = DatabricksSession.builder.serverless().getOrCreate()
ws    = WorkspaceClient()

UC_CATALOG = os.getenv("UC_CATALOG", "alexander_booth")
UC_SCHEMA  = os.getenv("UC_SCHEMA",  "dqx_baseball")

BRONZE     = f"{UC_CATALOG}.{UC_SCHEMA}_bronze.pitches_raw"
SILVER     = f"{UC_CATALOG}.{UC_SCHEMA}_silver.pitches"
QUARANTINE = f"{UC_CATALOG}.{UC_SCHEMA}_quarantine.pitches_quarantine"

print(f"Reading: {BRONZE}")
print(f"Silver:  {SILVER}")
print(f"Quar.:   {QUARANTINE}")

Reading: alexander_booth.dqx_baseball_bronze.pitches_raw
Silver:  alexander_booth.dqx_baseball_silver.pitches
Quar.:   alexander_booth.dqx_baseball_quarantine.pitches_quarantine


## Load YAML checks

In [2]:
checks_yaml_path = Path.cwd() / "checks.yml"
yaml_checks      = yaml.safe_load(checks_yaml_path.read_text())

yaml_status = DQEngine.validate_checks(yaml_checks)
print(f"YAML checks  ({len(yaml_checks)} rules): {yaml_status}")
assert not yaml_status.has_errors, yaml_status

YAML checks  (7 rules): No errors found


## Add a few Python-defined checks

Same engine, code-first style. Useful when:

- Rules are dynamically generated from another system
- You want IDE autocomplete & type hints
- You need to compose `check_func_kwargs` from runtime config

In [3]:
py_checks = [
    # Pitcher / batter IDs follow a known prefix pattern
    DQRowRule(
        name="pitcher_id_format",
        column="pitcher_id",
        check_func=check_funcs.regex_match,
        check_func_kwargs={"regex": r"^P\d{4}$"},
        criticality=Criticality.WARN.value,
    ),
    DQRowRule(
        name="batter_id_format",
        column="batter_id",
        check_func=check_funcs.regex_match,
        check_func_kwargs={"regex": r"^B\d{4}$"},
        criticality=Criticality.WARN.value,
    ),
    # Belt-and-braces SQL check — pitch_speed_mph must be present AND positive.
    # Complements the YAML is_in_range[40,110] rule with a different failure message.
    DQRowRule(
        name="speed_above_min",
        column="pitch_speed_mph",
        check_func=check_funcs.sql_expression,
        check_func_kwargs={
            "expression": "pitch_speed_mph IS NOT NULL AND pitch_speed_mph > 0",
            "msg": "pitch_speed_mph must be positive",
        },
        criticality=Criticality.ERROR.value,
    ),
]

# Note: validate_checks() expects metadata dicts, not DQRule objects.
# Python-defined rules are validated at apply time instead — instantiation already
# enforces the type contract.
print(f"Python checks: {len(py_checks)} rules built")

Python checks: 3 rules built


## Run the engine

In [4]:
dq_engine = DQEngine(ws)

df = spark.read.table(BRONZE)
print(f"Input rows: {df.count():,}")

# Pass 1: YAML metadata checks → split into valid / invalid
valid_yaml_df, invalid_yaml_df = dq_engine.apply_checks_by_metadata_and_split(df, yaml_checks)

# Pass 2: re-run on the rows that survived YAML checks, this time with the
# Python-object rules. DQX rejects DataFrames that already have _errors/_warnings,
# so drop them between passes.
valid_clean    = valid_yaml_df.drop("_errors", "_warnings")
valid_df, invalid_py_df = dq_engine.apply_checks_and_split(valid_clean, py_checks)

# Final invalid set = anything that failed either pass
invalid_df = invalid_yaml_df.unionByName(invalid_py_df, allowMissingColumns=True)

valid_count, invalid_count = valid_df.count(), invalid_df.count()
print(f"Valid:     {valid_count:,}")
print(f"Invalid:   {invalid_count:,}  ({invalid_count / (valid_count+invalid_count):.1%})")

Input rows: 50,000


Valid:     45,291
Invalid:   6,070  (11.8%)


In [5]:
# Peek at what the quarantine rows actually look like.
# _errors / _warnings are arrays of structs containing rule name, message, columns, function, etc.
invalid_df.select(
    "at_bat_id", "pitch_speed_mph", "batting_avg", "pitch_type", "_errors", "_warnings"
).show(8, truncate=False)

+----------+---------------+-----------+----------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|at_bat_id |pitch_speed_mph|batting_avg|pitch_type|_errors                                                                                                                                                                                                 

## Land outputs

INSERT OVERWRITE pattern — `MERGE` is unsupported via Databricks Connect.

In [6]:
# Strip the DQX result columns from the silver table — those belong on quarantine only.
valid_df.drop("_errors", "_warnings") \
    .write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(SILVER)

invalid_df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(QUARANTINE)

print(f"Wrote {SILVER}")
print(f"Wrote {QUARANTINE}")

Wrote alexander_booth.dqx_baseball_silver.pitches
Wrote alexander_booth.dqx_baseball_quarantine.pitches_quarantine


In [7]:
# Sanity: rule-level breakdown of what got rejected.
spark.sql(f"""
    WITH unnested AS (
        SELECT explode(_errors) AS e FROM {QUARANTINE}
        UNION ALL
        SELECT explode(_warnings) FROM {QUARANTINE}
    )
    SELECT e.name AS rule, COUNT(*) AS failures
    FROM unnested
    GROUP BY e.name
    ORDER BY failures DESC
""").show(truncate=False)

+---------------------------+--------+
|rule                       |failures|
+---------------------------+--------+
|batting_avg_not_in_range   |1342    |
|pitch_speed_in_range       |1295    |
|inning_not_in_range        |693     |
|batter_id_is_null_or_empty |677     |
|pitch_type_in_enum         |671     |
|not_game_date_current_date |663     |
|pitcher_id_is_null_or_empty|641     |
|at_bat_id_is_not_unique    |100     |
+---------------------------+--------+



Silver and quarantine populated. Continue with `04_dq_metrics_over_time`.